In [1]:
# Step 1: Import libraries and load the API key
import os
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()   # reads the .env file
api_key = os.getenv("TMDB_API_KEY")

print("Key loaded:", api_key is not None)

Key loaded: True


In [2]:
# Step 2: Test the API with one movie (Avengers: Endgame)
url = "https://api.themoviedb.org/3/movie/299534"
response = requests.get(url, params={"api_key": api_key})

print("Status code:", response.status_code)
print("Title:", response.json().get("title"))

Status code: 200
Title: Avengers: Endgame


In [3]:
# Step 3: Fetch all movies from the lab list
movie_ids = [0, 299534, 19995, 140607, 299536, 597, 135397,
             420818, 24428, 168259, 99861, 284054, 12445,
             181808, 330457, 351286, 109445, 321612, 260513]

movies = []   # empty list to collect each movie

for movie_id in movie_ids:
    url = "https://api.themoviedb.org/3/movie/" + str(movie_id)
    params = {"api_key": api_key, "append_to_response": "credits"}
    response = requests.get(url, params=params)

    if response.status_code == 200:
        movies.append(response.json())
        print("OK:", movie_id)
    else:
        print("Not found:", movie_id)

df = pd.DataFrame(movies)
print("Number of movies:", len(df))

Not found: 0
OK: 299534
OK: 19995
OK: 140607
OK: 299536
OK: 597
OK: 135397
OK: 420818
OK: 24428
OK: 168259
OK: 99861
OK: 284054
OK: 12445
OK: 181808
OK: 330457
OK: 351286
OK: 109445
OK: 321612
OK: 260513
Number of movies: 18


In [4]:
# Step 4: Drop irrelevant columns
columns_to_drop = ["adult", "imdb_id", "original_title", "video", "homepage"]
df = df.drop(columns=columns_to_drop)

print("Columns left:", len(df.columns))
print(df.columns.tolist())

Columns left: 23
['backdrop_path', 'belongs_to_collection', 'budget', 'genres', 'id', 'origin_country', 'original_language', 'overview', 'popularity', 'poster_path', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'softcore', 'spoken_languages', 'status', 'tagline', 'title', 'vote_average', 'vote_count', 'credits']


In [5]:
# Step 5: Extract names from JSON-like columns

def get_names(value, key="name"):
    # value is a list of dictionaries -> join the names with "|"
    if type(value) != list:
        return None
    names = []
    for item in value:
        names.append(item[key])
    return "|".join(names)

def get_collection(value):
    # value is one dictionary, or empty if the movie has no collection
    if type(value) == dict:
        return value["name"]
    return None

df["genres"] = df["genres"].apply(get_names)
df["production_countries"] = df["production_countries"].apply(get_names)
df["production_companies"] = df["production_companies"].apply(get_names)
df["spoken_languages"] = df["spoken_languages"].apply(get_names, key="english_name")
df["belongs_to_collection"] = df["belongs_to_collection"].apply(get_collection)

df[["title", "genres", "belongs_to_collection", "spoken_languages"]].head()

,title,genres,belongs_to_collection,spoken_languages
0,Avengers: Endgame,Adventure|Science Fiction|Action,The Avengers Collection,English|Japanese|Xhosa
1,Avatar,Science Fiction|Action|Adventure,Avatar Collection,English|Spanish
2,Star Wars: The Force Awakens,Adventure|Action|Science Fiction,Star Wars Collection,English
3,Avengers: Infinity War,Adventure|Action|Science Fiction,The Avengers Collection,English|Xhosa
4,Titanic,Drama|Romance,None,English|French|German|Swedish|Italian|Russian


In [6]:
# Step 6: Extract cast and director from credits

def get_cast(credits):
    # names of all actors, joined with "|"
    names = []
    for person in credits["cast"]:
        names.append(person["name"])
    return "|".join(names)

def get_director(credits):
    # find the crew member whose job is "Director"
    for person in credits["crew"]:
        if person["job"] == "Director":
            return person["name"]
    return None

df["cast"] = df["credits"].apply(get_cast)
df["cast_size"] = df["credits"].apply(lambda c: len(c["cast"]))
df["director"] = df["credits"].apply(get_director)
df["crew_size"] = df["credits"].apply(lambda c: len(c["crew"]))

df = df.drop(columns=["credits"])

df[["title", "director", "cast_size", "crew_size"]].head()

,title,director,cast_size,crew_size
0,Avengers: Endgame,Anthony Russo,103,613
1,Avatar,James Cameron,68,999
2,Star Wars: The Force Awakens,J.J. Abrams,99,270
3,Avengers: Infinity War,Joe Russo,69,737
4,Titanic,James Cameron,116,271


In [7]:
# Step 7: Fix data types and convert money to millions

# Make sure these are numbers (bad values become NaN)
df["budget"] = pd.to_numeric(df["budget"], errors="coerce")
df["revenue"] = pd.to_numeric(df["revenue"], errors="coerce")
df["id"] = pd.to_numeric(df["id"], errors="coerce")
df["popularity"] = pd.to_numeric(df["popularity"], errors="coerce")
df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")

# 0 is not a real budget, revenue or runtime -> treat it as missing
for column in ["budget", "revenue", "runtime"]:
    df[column] = df[column].replace(0, float("nan"))

# Convert to million USD and drop the old columns
df["budget_musd"] = df["budget"] / 1000000
df["revenue_musd"] = df["revenue"] / 1000000
df = df.drop(columns=["budget", "revenue"])

df[["title", "release_date", "budget_musd", "revenue_musd", "runtime"]].head()

,title,release_date,budget_musd,revenue_musd,runtime
0,Avengers: Endgame,2019-04-24,356.0,2927.449455,181
1,Avatar,2009-12-16,237.0,2923.706026,162
2,Star Wars: The Force Awakens,2015-12-15,245.0,2068.223624,136
3,Avengers: Infinity War,2018-04-25,300.0,2052.415039,149
4,Titanic,1997-12-18,200.0,2264.162353,194


In [8]:
# Step 8: Finish cleaning (lab points 6 to 9)

# Movies with 0 votes: their rating means nothing -> set it to missing
df.loc[df["vote_count"] == 0, "vote_average"] = float("nan")

# Placeholder text in overview and tagline -> missing
for column in ["overview", "tagline"]:
    df[column] = df[column].replace(["No Data", ""], float("nan"))

print("Rows at start:", len(df))

# Remove duplicate movies and rows with no id or title
df = df.drop_duplicates(subset="id")
df = df.dropna(subset=["id", "title"])
print("After duplicates and missing id/title:", len(df))

# Keep only rows with at least 10 filled-in columns
df = df.dropna(thresh=10)
print("After keeping rows with 10+ values:", len(df))

# Keep only released movies, then drop the status column
df = df[df["status"] == "Released"]
df = df.drop(columns=["status"])
print("After keeping released movies:", len(df))

Rows at start: 18
After duplicates and missing id/title: 18
After keeping rows with 10+ values: 18
After keeping released movies: 18


In [9]:
# Step 9: Reorder columns and reset the index (lab points 10 and 11)
column_order = ["id", "title", "tagline", "release_date", "genres", "belongs_to_collection",
                "original_language", "budget_musd", "revenue_musd", "production_companies",
                "production_countries", "vote_count", "vote_average", "popularity", "runtime",
                "overview", "spoken_languages", "poster_path", "cast", "cast_size", "director", "crew_size"]

df = df[column_order]
df = df.reset_index(drop=True)

print("Final shape (rows, columns):", df.shape)
df.head()

Final shape (rows, columns): (18, 22)


,id,title,tagline,release_date,genres,belongs_to_collection,original_language,budget_musd,revenue_musd,production_companies,...,vote_average,popularity,runtime,overview,spoken_languages,poster_path,cast,cast_size,director,crew_size
0,299534,Avengers: Endgame,Avenge the fallen.,2019-04-24,Adventure|Science Fiction|Action,The Avengers Collection,en,356.0,2927.449455,Marvel Studios,...,8.200,96.8577,181,After the devastating events of Avengers: Infi...,English|Japanese|Xhosa,/bR8ISy1O9XQxqiy0fQFw2BX72RQ.jpg,Robert Downey Jr.|Chris Evans|Mark Ruffalo|Chr...,103,Anthony Russo,613
1,19995,Avatar,Enter the world of Pandora.,2009-12-16,Science Fiction|Action|Adventure,Avatar Collection,en,237.0,2923.706026,Dune Entertainment|Lightstorm Entertainment|20...,...,7.600,41.8923,162,"In the 22nd century, a paraplegic Marine is di...",English|Spanish,/gKY6q7SjCkAU6FqvqWybDYgUKIF.jpg,Sam Worthington|Zoe Saldaña|Sigourney Weaver|S...,68,James Cameron,999
2,140607,Star Wars: The Force Awakens,Every generation has a story.,2015-12-15,Adventure|Action|Science Fiction,Star Wars Collection,en,245.0,2068.223624,Lucasfilm Ltd.|Bad Robot,...,7.200,25.1323,136,Thirty years after defeating the Galactic Empi...,English,/wqnLdwVXoBjKibFRR5U3y0aDUhs.jpg,Daisy Ridley|John Boyega|Oscar Isaac|Adam Driv...,99,J.J. Abrams,270
3,299536,Avengers: Infinity War,An entire universe. Once and for all.,2018-04-25,Adventure|Action|Science Fiction,The Avengers Collection,en,300.0,2052.415039,Marvel Studios,...,8.245,93.1399,149,As the Avengers and their allies have continue...,English|Xhosa,/7WsyChQLEftFiDOVTGkv3hFpyyt.jpg,Robert Downey Jr.|Chris Evans|Chris Hemsworth|...,69,Joe Russo,737
4,597,Titanic,Nothing on earth could come between them.,1997-12-18,Drama|Romance,None,en,200.0,2264.162353,Paramount Pictures|20th Century Fox|Lightstorm...,...,7.900,39.3321,194,101-year-old Rose DeWitt Bukater tells the sto...,English|French|German|Swedish|Italian|Russian,/9xjZS2rlVxm8SFx8kPC3aIGCOYQ.jpg,Leonardo DiCaprio|Kate Winslet|Billy Zane|Kath...,116,James Cameron,271


In [10]:
# Step 10: Add profit and ROI, and create a ranking function (UDF)

df["profit_musd"] = df["revenue_musd"] - df["budget_musd"]
df["roi"] = df["revenue_musd"] / df["budget_musd"]

def rank_movies(data, column, highest=True, top=5):
    # Sort movies by one column and show the top results
    if highest:
        result = data.sort_values(column, ascending=False)
    else:
        result = data.sort_values(column, ascending=True)
    return result[["title", column]].head(top)

# Test it: the 5 movies with the highest revenue
rank_movies(df, "revenue_musd")

,title,revenue_musd
0,Avengers: Endgame,2927.449455
1,Avatar,2923.706026
4,Titanic,2264.162353
2,Star Wars: The Force Awakens,2068.223624
3,Avengers: Infinity War,2052.415039


In [11]:
# Step 11: All 10 rankings using the UDF

# Smaller tables for the lab's conditions
big_budget = df[df["budget_musd"] >= 10]      # for ROI: only budget >= 10M
enough_votes = df[df["vote_count"] >= 10]     # for ratings: only >= 10 votes

print("1. Highest Revenue");    display(rank_movies(df, "revenue_musd"))
print("2. Highest Budget");     display(rank_movies(df, "budget_musd"))
print("3. Highest Profit");     display(rank_movies(df, "profit_musd"))
print("4. Lowest Profit");      display(rank_movies(df, "profit_musd", highest=False))
print("5. Highest ROI");        display(rank_movies(big_budget, "roi"))
print("6. Lowest ROI");         display(rank_movies(big_budget, "roi", highest=False))
print("7. Most Voted");         display(rank_movies(df, "vote_count"))
print("8. Highest Rated");      display(rank_movies(enough_votes, "vote_average"))
print("9. Lowest Rated");       display(rank_movies(enough_votes, "vote_average", highest=False))
print("10. Most Popular");      display(rank_movies(df, "popularity"))

1. Highest Revenue


,title,revenue_musd
0,Avengers: Endgame,2927.449455
1,Avatar,2923.706026
4,Titanic,2264.162353
2,Star Wars: The Force Awakens,2068.223624
3,Avengers: Infinity War,2052.415039


2. Highest Budget


,title,budget_musd
0,Avengers: Endgame,356.0
3,Avengers: Infinity War,300.0
12,Star Wars: The Last Jedi,300.0
6,The Lion King,260.0
2,Star Wars: The Force Awakens,245.0


3. Highest Profit


...

,title,vote_average
3,Avengers: Infinity War,8.245
0,Avengers: Endgame,8.200
11,Harry Potter and the Deathly Hallows: Part 2,8.100
7,The Avengers,8.080
4,Titanic,7.900


9. Lowest Rated


,title,vote_average
14,Jurassic World: Fallen Kingdom,6.500
12,Star Wars: The Last Jedi,6.700
5,Jurassic World,6.703
16,Beauty and the Beast,7.000
6,The Lion King,7.090


10. Most Popular


,title,popularity
0,Avengers: Endgame,96.8577
3,Avengers: Infinity War,93.1399
7,The Avengers,66.0038
9,Avengers: Age of Ultron,49.2888
16,Beauty and the Beast,45.3530


In [12]:
# Step 12: Advanced search queries

# Search 1: Science Fiction + Action movies starring Bruce Willis, best rated first
search1 = df[
    df["genres"].str.contains("Science Fiction", na=False)
    & df["genres"].str.contains("Action", na=False)
    & df["cast"].str.contains("Bruce Willis", na=False)
]
search1 = search1.sort_values("vote_average", ascending=False)
print("Search 1: Sci-Fi Action movies with Bruce Willis")
display(search1[["title", "genres", "vote_average"]])

# Search 2: movies starring Uma Thurman, directed by Quentin Tarantino, shortest first
search2 = df[
    df["cast"].str.contains("Uma Thurman", na=False)
    & (df["director"] == "Quentin Tarantino")
]
search2 = search2.sort_values("runtime", ascending=True)
print("Search 2: Uma Thurman + Quentin Tarantino")
display(search2[["title", "director", "runtime"]])

Search 1: Sci-Fi Action movies with Bruce Willis


,title,genres,vote_average


Search 2: Uma Thurman + Quentin Tarantino


,title,director,runtime


**Note on the search results:** Both searches return no rows. The code works, but the 18 movies in the lab's list do not include any Science Fiction Action movie with Bruce Willis, or any Quentin Tarantino movie with Uma Thurman. With a larger dataset, the same code would return the matching movies.

In [13]:
# Step 13: Franchise vs standalone movies

# A movie is a franchise movie if belongs_to_collection is not empty
df["is_franchise"] = df["belongs_to_collection"].notna()

franchise = df[df["is_franchise"] == True]
standalone = df[df["is_franchise"] == False]

comparison = pd.DataFrame({
    "Franchise": [franchise["revenue_musd"].mean(),
                  franchise["roi"].median(),
                  franchise["budget_musd"].mean(),
                  franchise["popularity"].mean(),
                  franchise["vote_average"].mean()],
    "Standalone": [standalone["revenue_musd"].mean(),
                   standalone["roi"].median(),
                   standalone["budget_musd"].mean(),
                   standalone["popularity"].mean(),
                   standalone["vote_average"].mean()]
}, index=["Mean Revenue (M$)", "Median ROI", "Mean Budget (M$)", "Mean Popularity", "Mean Rating"])

print("Franchise movies:", len(franchise), "| Standalone movies:", len(standalone))
comparison.round(2)

Franchise movies: 16 | Standalone movies: 2


,Franchise,Standalone
Mean Revenue (M$),1690.78,1765.14
Median ROI,7.84,9.62
Mean Budget (M$),218.00,180.00
Mean Popularity,41.90,42.34
Mean Rating,7.38,7.45


In [14]:
# Step 14: Most successful franchises and directors

# Franchises: group the franchise movies by collection name
franchise_stats = franchise.groupby("belongs_to_collection").agg(
    movies=("id", "count"),
    total_budget=("budget_musd", "sum"),
    mean_budget=("budget_musd", "mean"),
    total_revenue=("revenue_musd", "sum"),
    mean_revenue=("revenue_musd", "mean"),
    mean_rating=("vote_average", "mean"),
)
franchise_stats = franchise_stats.sort_values("total_revenue", ascending=False)
print("Most successful franchises")
display(franchise_stats.round(2))

# Directors: group all movies by director
director_stats = df.groupby("director").agg(
    movies=("id", "count"),
    total_revenue=("revenue_musd", "sum"),
    mean_rating=("vote_average", "mean"),
)
director_stats = director_stats.sort_values("total_revenue", ascending=False)
print("Most successful directors")
display(director_stats.round(2))

Most successful franchises


,movies,total_budget,mean_budget,total_revenue,mean_revenue,mean_rating
belongs_to_collection,,,,,,
The Avengers Collection,4,1111.0,277.75,7904.08,1976.02,7.95
Star Wars Collection,2,545.0,272.50,3402.63,1701.32,6.95
Jurassic Park Collection,2,320.0,160.00,2982.01,1491.00,6.60
Avatar Collection,1,237.0,237.00,2923.71,2923.71,7.60
Frozen Collection,2,300.0,150.00,2727.90,1363.95,7.21
The Lion King (Reboot) Collection,1,260.0,260.00,1662.02,1662.02,7.09
The Fast and the Furious Collection,1,190.0,190.00,1515.40,1515.40,7.21
Black Panther Collection,1,200.0,200.00,1349.93,1349.93,7.36
Harry Potter Collection,1,125.0,125.00,1341.51,1341.51,8.10


Most successful directors


,movies,total_revenue,mean_rating
director,,,
James Cameron,2,5187.87,7.75
Anthony Russo,1,2927.45,8.20
Joss Whedon,2,2924.22,7.68
J.J. Abrams,1,2068.22,7.20
Joe Russo,1,2052.42,8.24
Colin Trevorrow,1,1671.54,6.70
Jon Favreau,1,1662.02,7.09
James Wan,1,1515.40,7.21
Jennifer Lee,1,1453.68,7.22
